# HTGNN Forex XAI Workflow

This notebook shows the reproducible workflow used for the report: data preparation, HTGNN training, backtesting, XAI execution, and visual inspection of every figure discussed in the main text and appendix. Command cells start with `!` so they run in the terminal from Jupyter.

In [ ]:
from pathlib import Path
import os
from IPython.display import SVG, Image, display


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')


ROOT = find_repo_root()
os.chdir(ROOT)


def project_path(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


def latest_dir(root):
    root = project_path(root)
    candidates = [path for path in root.iterdir() if path.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No run directories found under {root}')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def display_svg(path):
    display(SVG(filename=str(project_path(path))))


def display_png(path, width=None):
    display(Image(filename=str(project_path(path)), width=width))


ROOT


## 1. Data preparation

Run these cells when starting from an empty `data/` directory. The first command downloads the Yahoo Finance market series; the second command applies the configured preprocessing, including the centred currency-return transform used by the final explanations.

In [ ]:
!python -m src.data.download --config configs/download.yaml

In [ ]:
!python -m src.data.transform --config configs/transform.yaml

## 2. Train the model

The main explained model is the HTGNN checkpoint. It encodes each market block as a temporal node, applies relation-aware message passing, and reads the allocation from the `portfolio_signal` node.

In [ ]:
!python -m src.train --config configs/models/pointwise/HTGNN.yaml

The architecture figure in the report summarises this pipeline: temporal node encoders first compress each input block, message passing then exchanges information over the heterogeneous graph, and the final head converts the portfolio node state into long-only allocation weights.

In [ ]:
display_png('report/figures/graphics/htgnn_architecture.png', width=620)

## 3. Run the backtest

The backtest applies the trained checkpoint to the configured strategy-evaluation window and writes accumulated returns, allocation summaries, and metrics under `backtests/`.

In [ ]:
!python -m src.backtest --checkpoint latest --config configs/backtest.yaml

## 4. Visualise backtest results

The helper below loads generated plots. The report uses a chronological split: train from 2018-01-01 to 2023-12-29, validation from 2024-01-01 to 2024-12-31, and test from 2025-01-02 to 2025-12-31. The displayed backtest covers the configured 2024-01-01 to 2025-12-31 strategy-evaluation window.

In [ ]:
backtest_dir = latest_dir("backtests")
backtest_dir

The accumulated-return curve is the main performance view. In the report, the HTGNN separates clearly from USD and from the best single-currency benchmark, while the table records the best single FX benchmark as CHF.

In [ ]:
display_svg(backtest_dir / 'accumulated_returns.svg')

The allocation summary rules out the trivial explanation that the model simply holds one currency. The reported checkpoint is diversified, with larger average exposure to AUD and JPY but meaningful allocation across the basket.

In [ ]:
display_svg(backtest_dir / 'allocations_summary.svg')

## 5. Run the XAI pipeline

This command regenerates the XAI artifacts used in the report: portfolio-signal SHAP, Integrated Gradients, temporal and node-wise occlusion, and deletion/insertion faithfulness checks.

In [ ]:
!python -m xai.xai --checkpoint latest --model auto --split test --target variance --output-dir xai --device cpu --max-samples 260 --background-samples 32 --n-local 3 --top-k 5 --seed 42 --methods all

## 6. Visualise XAI results

The absolute SHAP grid for predicted mean allocations focuses on the `portfolio_signal` node. The strongest effects concentrate in the AUD output, which is consistent with AUD being the largest average allocation in the backtest. Because the surrogate has low fidelity, this should be read as a restricted portfolio-signal view, not as a full explanation of the HTGNN.

In [ ]:
display_svg('xai/global_explanations/portfolio_signal_shap_mean.svg')

The signed SHAP grid for mean allocations is almost zero everywhere. The contrast with the absolute grid is informative: the portfolio-signal inputs matter in magnitude, but their net direction changes across samples and cancels out on average.

In [ ]:
display_svg('xai/global_explanations/portfolio_signal_shap_mean_signed.svg')

The absolute SHAP grid for predicted marginal allocation uncertainty asks a different question: which portfolio-signal currencies affect the model's variance output. In the report this figure is kept in the appendix because the central discussion focuses on mean allocations and performance.

In [ ]:
display_svg('xai/global_explanations/portfolio_signal_shap_variance.svg')

The signed variance SHAP grid keeps direction. As with the signed mean grid, near-zero signed values mainly indicate cancellation of positive and negative effects, not necessarily absence of sensitivity.

In [ ]:
display_svg('xai/global_explanations/portfolio_signal_shap_variance_signed.svg')

Integrated Gradients gives the full-graph view. The graph highlights five especially important nodes: `fx_future`, `commodity_future`, `portfolio_signal`, `fx_usd_pair`, and `commodity_etf`. This supports the interpretation that the HTGNN relies on direct FX derivatives, commodity-linked context, and recent portfolio-regime information.

In [ ]:
display_svg('xai/local_explanations/integrated_gradients_graph.svg')

The node-importance histogram shows the same ranking with variance intervals. Those intervals are small, so the local examples used for IG do not suggest a highly unstable node-attention pattern.

In [ ]:
display_svg('xai/local_explanations/integrated_gradients_node_importance.svg')

The last-message IG graph starts attribution from the states before the final message passing layer. It checks whether the generic latent nodes contribute after message passing. They are not exactly zero, but their focus remains negligible, which suggests that the model is not using those generic nodes effectively to model useful cross-feature information.

In [ ]:
display_svg('xai/local_explanations/integrated_gradients_last_message_graph.svg')

The feature-level pie charts decompose IG inside each observed node. This is useful for the leakage check: highly attended nodes do not appear to depend on a single dominant feature, so a one-feature leakage explanation is less plausible.

In [ ]:
display_svg('xai/local_explanations/input_attention_pie_charts.svg')

Deletion/insertion relative to the full model is the main faithfulness check used in the report. Removing the first few important nodes only modestly reduces retained performance, while the larger break appears once `fx_future` is also removed. This points to joint dependence and partial redundancy among the top market blocks.

In [ ]:
display_svg('xai/evaluation/deletion_insertion_vs_full_model_curve.svg')

Global temporal occlusion replaces five lookback windows with the background baseline. The latest window is the most helpful one: masking it lowers performance. By contrast, the 12-16 window can improve performance when masked, suggesting that some mid-window information is harmful on the evaluated period.

In [ ]:
display_svg('xai/local_explanations/temporal_occlusion_windows.svg')

Node-wise temporal occlusion applies those same five windows one node at a time. The important negative nuance is `commodity_future`: it receives high IG focus, but occluding it does not robustly reduce performance and can even improve the performance ratio in later windows. That makes it influential but potentially noisy or harmful in part of the test period.

In [ ]:
display_svg('xai/local_explanations/node_temporal_occlusion_windows.svg')